#Initialization

In [0]:
%sql
USE CATALOG workspace

# Read Bronze table

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df AS
SELECT * FROM workspace.bronze.crm_sales_details

#Silver Transformations

##Trimming

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_trimmed AS
SELECT TRIM(sls_ord_num) AS sls_ord_num,
       TRIM(sls_prd_key) AS sls_prd_key,
       sls_cust_id, sls_order_dt, sls_ship_dt, sls_due_dt,
       sls_sales, sls_quantity, sls_price
FROM df

##Cleaning Dates

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_dates AS
SELECT sls_ord_num, sls_prd_key, sls_cust_id,
  CASE WHEN sls_order_dt = 0 OR LENGTH(sls_order_dt) != 8 THEN NULL
       ELSE TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd') END AS sls_order_dt,
  CASE WHEN sls_ship_dt = 0 OR LENGTH(sls_ship_dt) != 8 THEN NULL
       ELSE TO_DATE(CAST(sls_ship_dt AS STRING), 'yyyyMMdd') END AS sls_ship_dt,
  CASE WHEN sls_due_dt = 0 OR LENGTH(sls_due_dt) != 8 THEN NULL
       ELSE TO_DATE(CAST(sls_due_dt AS STRING), 'yyyyMMdd') END AS sls_due_dt,
  sls_sales, sls_quantity, sls_price
FROM df_trimmed

##Sales and Price Corrections

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_price AS
SELECT sls_ord_num, sls_prd_key, sls_cust_id, sls_order_dt, sls_ship_dt, sls_due_dt,
       sls_sales, sls_quantity,
  CASE WHEN sls_price IS NULL OR sls_price <= 0 THEN
         CASE WHEN sls_quantity != 0 THEN sls_sales / sls_quantity ELSE NULL END
       ELSE sls_price END AS sls_price
FROM df_dates

## Renaming Columns

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW df_renamed AS
SELECT sls_ord_num  AS order_number,
       sls_prd_key  AS product_number,
       sls_cust_id  AS customer_id,
       sls_order_dt AS order_date,
       sls_ship_dt  AS ship_date,
       sls_due_dt   AS due_date,
       sls_sales    AS sales_amount,
       sls_quantity AS quantity,
       sls_price    AS price
FROM df_price

## Sanity checks of dataframe

In [0]:
%sql
SELECT * FROM df_renamed LIMIT 10

#Writing Silver Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.crm_sales USING DELTA AS
SELECT * FROM df_renamed

## Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 10